In [1]:
import torch
import torch.nn as nn
import numpy as np
import math
import time

# Load frozen embedding table
embedding_table = np.load("tokenizer_v3/token_embeddings.npy")
embedding_tensor = torch.tensor(embedding_table, dtype=torch.float32)

VOCAB_SIZE    = embedding_tensor.shape[0]   # 4096
MINILM_DIM    = embedding_tensor.shape[1]   # 384
print(f"Vocab size : {VOCAB_SIZE}")
print(f"MiniLM dim : {MINILM_DIM}")


Vocab size : 16384
MiniLM dim : 384


In [2]:
import torch.nn.functional as F 

DIM      = 384
EXPANDED_DIM = 256
N_HEADS  = 8
N_LAYERS = 8
FFN_DIM  = 1024
HEAD_DIM = DIM // N_HEADS 

def sinusoidal_encoding(seq_len, dim, device):
    pe       = torch.zeros(seq_len, dim, device=device)
    position = torch.arange(seq_len, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # (seq_len, dim)

class LayerNorm(nn.Module):
    """
    y = ((x - mean) / sqrt(var + eps)) * gamma + beta
    gamma, beta are learned per-feature scalars — shape (dim,)
    """
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(dim))   # scale
        self.beta  = nn.Parameter(torch.zeros(dim))  # shift

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)           # (B, T, 1)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)  # (B, T, 1)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)    # (B, T, D)
        return self.gamma * x_norm + self.beta               # (B, T, D)

class MultiHeadCausalAttention(nn.Module):
    """
    Projects input into Q, K, V — splits into H heads — computes scaled
    dot-product attention with a causal mask — concatenates heads — projects out.

    Q = x W_q      shape: (B, T, D)
    K = x W_k      shape: (B, T, D)
    V = x W_v      shape: (B, T, D)

    Reshape to (B, H, T, head_dim), then:
        scores = Q @ Kᵀ / sqrt(head_dim)    (B, H, T, T)
        scores = scores + causal_mask        (upper triangle = -inf)
        weights = softmax(scores, dim=-1)    (B, H, T, T)
        out = weights @ V                    (B, H, T, head_dim)

    Concat heads → (B, T, D), project out via W_o
    """
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads  = n_heads
        self.head_dim = dim // n_heads       # 48
        self.scale    = self.head_dim ** -0.5

        self.W_q = nn.Linear(dim, dim, bias=False)
        self.W_k = nn.Linear(dim, dim, bias=False)
        self.W_v = nn.Linear(dim, dim, bias=False)
        self.W_o = nn.Linear(dim, dim, bias=False)

        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape

        # --- Project & split into heads ---
        Q = self.W_q(x)  # (B, T, D)
        K = self.W_k(x)  # (B, T, D)
        V = self.W_v(x)  # (B, T, D)

        # Reshape: (B, T, D) → (B, H, T, head_dim)
        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # --- Scaled dot-product attention ---
        # (B, H, T, head_dim) @ (B, H, head_dim, T) → (B, H, T, T)
        scores = (Q @ K.transpose(-2, -1)) * self.scale

        # Causal mask: positions can only attend to themselves and earlier tokens
        # Upper triangle (future tokens) set to -inf → softmax drives them to 0
        causal_mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1
        )
        scores = scores.masked_fill(causal_mask, float('-inf'))

        weights = torch.softmax(scores, dim=-1)  # (B, H, T, T)
        weights = self.attn_drop(weights)

        # (B, H, T, T) @ (B, H, T, head_dim) → (B, H, T, head_dim)
        out = weights @ V

        # --- Concat heads & project ---
        # (B, H, T, head_dim) → (B, T, H*head_dim) = (B, T, D)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)  # (B, T, D)

class FeedForward(nn.Module):
    """
    Two-layer MLP with GELU activation.
    FFN(x) = GELU(x W_1 + b_1) W_2 + b_2

    Expands dim → ffn_dim (wider representation),
    then projects back down ffn_dim → dim.
    """
    def __init__(self, dim, ffn_dim, dropout=0.1):
        super().__init__()
        self.W_1 = nn.Linear(dim, ffn_dim)       # expand
        self.W_2 = nn.Linear(ffn_dim, dim)       # contract
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = F.gelu(self.W_1(x))   # (B, T, ffn_dim)
        x = self.drop(x)
        x = self.W_2(x)           # (B, T, dim)
        return x

class TransformerBlock(nn.Module):
    """
    Pre-norm residual block (norm_first=True, same as your original config).

    x = x + Attention(LayerNorm(x))   ← self-attention sub-layer
    x = x + FFN(LayerNorm(x))         ← feed-forward sub-layer

    Pre-norm (LN before the sub-layer) stabilises training at depth
    vs post-norm (LN after the residual add).
    """
    def __init__(self, dim, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.norm_1 = LayerNorm(dim)
        self.attn   = MultiHeadCausalAttention(dim, n_heads, dropout)
        self.norm_2 = LayerNorm(dim)
        self.ffn    = FeedForward(dim, ffn_dim, dropout)
        self.drop   = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm_1(x)))  # (B, T, D)
        x = x + self.drop(self.ffn(self.norm_2(x)))   # (B, T, D)
        return x

class MicroLM(nn.Module):
    """
    Full architecture:
      1. Frozen MiniLM embedding lookup         → (B, T, 384)
      2. Sinusoidal positional encoding added   → (B, T, 384)
      3. N_LAYERS x TransformerBlock            → (B, T, 384)
      4. Final LayerNorm                        → (B, T, 384)
      5. Linear output head → logits            → (B, T, VOCAB_SIZE)
    """
    def __init__(self):
        super().__init__()
        self.register_buffer("embedding_table", embedding_tensor)

        # Expander 384 → 512 (trainable)
        self.expander = nn.Sequential(
            nn.Linear(MINILM_DIM, EXPANDED_DIM),
            nn.LayerNorm(EXPANDED_DIM),
        )
        
        self.blocks = nn.ModuleList([
            TransformerBlock(EXPANDED_DIM, N_HEADS, FFN_DIM, dropout=0.1)
            for _ in range(N_LAYERS)
        ])

        self.norm        = LayerNorm(EXPANDED_DIM)
        self.output_head = nn.Linear(EXPANDED_DIM, VOCAB_SIZE, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape

        x = self.embedding_table[token_ids]                    # (B, T, 512)
        x = self.expander(x)
        x = x + sinusoidal_encoding(T, EXPANDED_DIM, token_ids.device) # (B, T, 512)

        for block in self.blocks:
            x = block(x)   # (B, T, 512)

        x = self.norm(x)
        return self.output_head(x)  # (B, T, VOCAB_SIZE)

In [4]:
model     = MicroLM()
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")

# print(f"\nParam breakdown:")
# for name, p in model.named_parameters():
#     if p.requires_grad:
#         print(f"  {name:55s} {p.numel():>10,}")

# Forward pass
x      = torch.randint(0, VOCAB_SIZE, (2, 32))
logits = model(x)
print(f"\nInput  : {x.shape}")
print(f"Output : {logits.shape}")

Trainable params : 10,603,776
Frozen params    : 0  (embedding table)
Total params     : 10,603,776

Input  : torch.Size([2, 32])
Output : torch.Size([2, 32, 16384])


In [5]:
# Freeze expander
for param in model.expander.parameters():
    param.requires_grad = False

In [6]:
total     = sum(p.numel() for p in model.expander.parameters())
trainable = sum(p.numel() for p in model.expander.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")


Trainable params : 0
Frozen params    : 99,072  (embedding table)
Total params     : 99,072


In [5]:
(2*268435456)/10603776

50.63016344366384

In [7]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
import json

# ──────────────────────────────────────────────────────────────────────────────
# tokenizer
# ──────────────────────────────────────────────────────────────────────────────
tok = Tokenizer.from_file("tokenizer_v3/tokenizer.json")

PAD_ID       = 0
UNK_ID       = 1
BOS_ID       = 2
EOS_ID       = 3
SYSTEM_ID    = 4
USER_ID      = 5
ASSISTANT_ID = 6

ROLE_TO_ID = {
    "system": SYSTEM_ID,
    "user": USER_ID,
    "assistant": ASSISTANT_ID,
}

def encode(text):
    return tok.encode(text).ids


# ──────────────────────────────────────────────────────────────────────────────
# convert messages → token ids
# ──────────────────────────────────────────────────────────────────────────────
def messages_to_tokens(messages):
    ids = [BOS_ID]

    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        ids.append(ROLE_TO_ID[role])
        ids.extend(encode(content))

    ids.append(EOS_ID)

    return ids


# ──────────────────────────────────────────────────────────────────────────────
# dataset
# ──────────────────────────────────────────────────────────────────────────────
MAX_SEQ = 512

class ConversationDataset(Dataset):
    def __init__(self, jsonl_path):
        self.samples = []

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)

                ids = messages_to_tokens(row["messages"])

                # truncate
                ids = ids[:MAX_SEQ]

                if len(ids) < 2:
                    continue

                self.samples.append(torch.tensor(ids, dtype=torch.long))

        print(f"Loaded {len(self.samples):,} conversations")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ──────────────────────────────────────────────────────────────────────────────
# collate
# full-sequence training (your current preference)
# ──────────────────────────────────────────────────────────────────────────────
def collate_fn(batch):
    max_len = max(len(x) for x in batch)

    x_batch = []
    y_batch = []

    for seq in batch:
        x = seq[:-1]
        y = seq[1:]

        pad_len = max_len - 1 - len(x)

        if pad_len > 0:
            x = torch.cat([
                x,
                torch.full((pad_len,), PAD_ID, dtype=torch.long)
            ])

            y = torch.cat([
                y,
                torch.full((pad_len,), -100, dtype=torch.long)
            ])

        x_batch.append(x)
        y_batch.append(y)

    x_batch = torch.stack(x_batch)
    y_batch = torch.stack(y_batch)

    return x_batch, y_batch


# ──────────────────────────────────────────────────────────────────────────────
# dataset + loader
# ──────────────────────────────────────────────────────────────────────────────
dataset = ConversationDataset("conversation.jsonl")


Loaded 201,526 conversations


In [8]:
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)


In [9]:
# ──────────────────────────────────────────────────────────────────────────────
# model
# your existing model class must already exist:
# class MicroLM(...)
# ──────────────────────────────────────────────────────────────────────────────
device = "cuda"

model = MicroLM().to(device)
model

MicroLM(
  (expander): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (blocks): ModuleList(
    (0-7): 8 x TransformerBlock(
      (norm_1): LayerNorm()
      (attn): MultiHeadCausalAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=False)
        (W_k): Linear(in_features=256, out_features=256, bias=False)
        (W_v): Linear(in_features=256, out_features=256, bias=False)
        (W_o): Linear(in_features=256, out_features=256, bias=False)
        (attn_drop): Dropout(p=0.1, inplace=False)
      )
      (norm_2): LayerNorm()
      (ffn): FeedForward(
        (W_1): Linear(in_features=256, out_features=1024, bias=True)
        (W_2): Linear(in_features=1024, out_features=256, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
      (drop): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm()
  (output_head): Linear(in_features=256, out_features=

In [10]:
# ──────────────────────────────────────────────────────────────────────────────
# load checkpoint
# ──────────────────────────────────────────────────────────────────────────────
checkpoint = torch.load(
    "checkpoints_v6/epoch_3.pt",
    map_location=device
)

model.load_state_dict(checkpoint["model"])

print("Loaded checkpoint successfully")

Loaded checkpoint successfully


In [12]:
# Freeze expander
for param in model.expander.parameters():
    param.requires_grad = False
    
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")

Trainable params : 10,504,704
Frozen params    : 99,072  (embedding table)
Total params     : 10,603,776


In [13]:
# ──────────────────────────────────────────────────────────────────────────────
# optimizer
# lower LR for SFT
# ──────────────────────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-3,
    weight_decay=0.01
)


# optionally restore optimizer
# if "optimizer_state_dict" in checkpoint:
#     optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
#     print("Loaded optimizer state")


# ──────────────────────────────────────────────────────────────────────────────
# training loop
# ──────────────────────────────────────────────────────────────────────────────
EPOCHS = 3

model.train()

for epoch in range(EPOCHS):

    running_loss = 0.0

    for step, (x, y) in enumerate(loader):

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1),
            ignore_index=-100
        )

        optimizer.zero_grad(set_to_none=True)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        running_loss += loss.item()

        if step % 100 == 0:
            avg = running_loss / (step + 1)

            print(
                f"epoch={epoch} "
                f"step={step} "
                f"loss={avg:.4f}"
            )

    # save checkpoint
    save_path = f"checkpoints_sft/epoch_{epoch}.pt"

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
    }, save_path)

    print(f"saved → {save_path}")

epoch=0 step=0 loss=7.4685
epoch=0 step=100 loss=6.6047
epoch=0 step=200 loss=5.9282
epoch=0 step=300 loss=5.6114
epoch=0 step=400 loss=5.4059
epoch=0 step=500 loss=5.2528
epoch=0 step=600 loss=5.1423
epoch=0 step=700 loss=5.0436
epoch=0 step=800 loss=4.9637
epoch=0 step=900 loss=4.8927
epoch=0 step=1000 loss=4.8314
epoch=0 step=1100 loss=4.7809
epoch=0 step=1200 loss=4.7337
epoch=0 step=1300 loss=4.6914
epoch=0 step=1400 loss=4.6511
epoch=0 step=1500 loss=4.6157
epoch=0 step=1600 loss=4.5832
epoch=0 step=1700 loss=4.5543
epoch=0 step=1800 loss=4.5247
epoch=0 step=1900 loss=4.4996
epoch=0 step=2000 loss=4.4733
epoch=0 step=2100 loss=4.4516
epoch=0 step=2200 loss=4.4292
epoch=0 step=2300 loss=4.4081
epoch=0 step=2400 loss=4.3869
epoch=0 step=2500 loss=4.3682
epoch=0 step=2600 loss=4.3508
epoch=0 step=2700 loss=4.3325
epoch=0 step=2800 loss=4.3153
epoch=0 step=2900 loss=4.2985
epoch=0 step=3000 loss=4.2831
epoch=0 step=3100 loss=4.2685
epoch=0 step=3200 loss=4.2536
epoch=0 step=3300 loss

In [20]:

PAD_ID       = 0
BOS_ID       = 2
EOS_ID       = 3
SYSTEM_ID    = 4
USER_ID      = 5
ASSISTANT_ID = 6

ID_TO_SPECIAL = {
    PAD_ID: "<PAD>",
    BOS_ID: "<BOS>",
    EOS_ID: "<EOS>",
    SYSTEM_ID: "<SYSTEM>",
    USER_ID: "<USER>",
    ASSISTANT_ID: "<ASSISTANT>",
}

def encode(text):
    return tok.encode(text).ids

def decode(ids):
    clean = []
    for i in ids:
        if i in ID_TO_SPECIAL:
            continue
        clean.append(i)
    return tok.decode(clean)


# ── load model ───────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

print("Model loaded.")


# ── build conversation prompt ────────────────────────────────────────────────
def build_prompt(messages):
    ids = [BOS_ID]

    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        if role == "system":
            ids.append(SYSTEM_ID)
        elif role == "user":
            ids.append(USER_ID)
        elif role == "assistant":
            ids.append(ASSISTANT_ID)
        else:
            raise ValueError(f"Unknown role: {role}")

        ids.extend(encode(content))

    # add assistant tag so model knows it should answer now
    ids.append(ASSISTANT_ID)

    return ids


# ── generation ───────────────────────────────────────────────────────────────
@torch.no_grad()
def generate_reply(
    messages,
    max_new_tokens=120,
    temperature=0.8,
    top_k=1,
):
    ids = build_prompt(messages)

    for _ in range(max_new_tokens):
        x = torch.tensor([ids], dtype=torch.long, device=device)

        logits = model(x)
        next_logits = logits[0, -1, :]

        # prevent generating weird control tokens too early if you want
        next_logits[PAD_ID] = -float("inf")
        next_logits[BOS_ID] = -float("inf")
        next_logits[USER_ID] = -float("inf")
        next_logits[SYSTEM_ID] = -float("inf")

        next_logits = next_logits / temperature

        if top_k is not None:
            values, indices = torch.topk(next_logits, top_k)
            filtered = torch.full_like(next_logits, -float("inf"))
            filtered[indices] = values
            next_logits = filtered

        probs = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()

        if next_id == EOS_ID:
            break

        # stop if model starts another role
        if next_id in [USER_ID, SYSTEM_ID, ASSISTANT_ID]:
            break

        ids.append(next_id)

    # only decode generated assistant part
    prompt_len = len(build_prompt(messages))
    generated = ids[prompt_len:]

    return decode(generated), ids


Model loaded.


In [40]:
%%time
# ── test single turn ─────────────────────────────────────────────────────────
messages = [
    {
        "role": "system",
        "content": "Your name is Lola. you are girl"
    },
    {
        "role": "user",
        "content": "my name is kunal"
    }
]

reply, token_ids = generate_reply(messages)

print("\nAssistant:")
print(reply)


Assistant:
She is Lola.
CPU times: user 63.2 ms, sys: 16.3 ms, total: 79.6 ms
Wall time: 78.2 ms
